In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🔍 Reto Semana 9: SecureBank Fraud Detection\n",
    "\n",
    "## Sistema de Detección de Anomalías en Transacciones con NumPy\n",
    "\n",
    "---\n",
    "\n",
    "### 📋 Contexto del Proyecto\n",
    "\n",
    "**SecureBank** es uno de los bancos digitales más grandes de México. Has sido contratado como **Analista de Riesgos Junior** para desarrollar un sistema básico de detección de transacciones anómalas que podrían indicar fraude.\n",
    "\n",
    "```\n",
    "╔══════════════════════════════════════════════════════════════════╗\n",
    "║                    SECUREBANK FRAUD DETECTION                    ║\n",
    "║                Sistema de Detección de Anomalías                 ║\n",
    "╠══════════════════════════════════════════════════════════════════╣\n",
    "║                                                                  ║\n",
    "║    💳 Transacciones analizadas: 500 por categoría                ║\n",
    "║    📊 Categorías: 5 tipos de comercio                            ║\n",
    "║    🎯 Objetivo: Detectar transacciones sospechosas               ║\n",
    "║    📈 Métodos: IQR y Z-Score                                     ║\n",
    "║                                                                  ║\n",
    "╚══════════════════════════════════════════════════════════════════╝\n",
    "```\n",
    "\n",
    "### 🎯 Objetivos de Aprendizaje\n",
    "\n",
    "En este reto aplicarás:\n",
    "- Cálculo de percentiles y cuartiles\n",
    "- Detección de outliers con método IQR\n",
    "- Detección de outliers con Z-Score\n",
    "- Análisis estadístico por categorías\n",
    "- Generación de reportes de anomalías\n",
    "\n",
    "---"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📦 Configuración Inicial\n",
    "\n",
    "Ejecuta esta celda para cargar NumPy y preparar el entorno."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "\n",
    "# Configuración para reproducibilidad\n",
    "np.random.seed(2024)\n",
    "\n",
    "# Configuración de impresión\n",
    "np.set_printoptions(precision=2, suppress=True)\n",
    "\n",
    "print(\"✅ NumPy cargado correctamente\")\n",
    "print(f\"   Versión: {np.__version__}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## 📊 Datos del Reto\n",
    "\n",
    "### Estructura de los Datos\n",
    "\n",
    "```\n",
    "CATEGORÍAS DE TRANSACCIONES\n",
    "════════════════════════════════════════════════════════════════\n",
    "\n",
    "Índice │ Categoría           │ Monto Típico    │ Descripción\n",
    "───────┼─────────────────────┼─────────────────┼───────────────\n",
    "   0   │ Supermercados       │ $200 - $2,000   │ Compras diarias\n",
    "   1   │ Restaurantes        │ $100 - $800     │ Alimentos\n",
    "   2   │ Gasolineras         │ $300 - $1,500   │ Combustible\n",
    "   3   │ Tiendas Online      │ $150 - $5,000   │ E-commerce\n",
    "   4   │ Entretenimiento     │ $50 - $500      │ Cine, streaming\n",
    "\n",
    "════════════════════════════════════════════════════════════════\n",
    "\n",
    "⚠️ TIPOS DE ANOMALÍAS SIMULADAS:\n",
    "─────────────────────────────────────────────────────────────────\n",
    "• Montos inusualmente altos (posible fraude)\n",
    "• Montos inusualmente bajos (posibles pruebas de tarjeta)\n",
    "• Patrones atípicos por categoría\n",
    "─────────────────────────────────────────────────────────────────\n",
    "```\n",
    "\n",
    "Ejecuta la siguiente celda para generar los datos de transacciones."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ═══════════════════════════════════════════════════════════════════\n",
    "#                  GENERACIÓN DE DATOS DE TRANSACCIONES\n",
    "# ═══════════════════════════════════════════════════════════════════\n",
    "\n",
    "np.random.seed(2024)\n",
    "\n",
    "# Configuración por categoría\n",
    "categorias = ['Supermercados', 'Restaurantes', 'Gasolineras', 'Tiendas_Online', 'Entretenimiento']\n",
    "n_categorias = len(categorias)\n",
    "\n",
    "# Parámetros de distribución por categoría (media, desv_std)\n",
    "params_categorias = {\n",
    "    'Supermercados': (800, 400),\n",
    "    'Restaurantes': (350, 150),\n",
    "    'Gasolineras': (700, 250),\n",
    "    'Tiendas_Online': (1200, 800),\n",
    "    'Entretenimiento': (200, 100)\n",
    "}\n",
    "\n",
    "n_transacciones_por_cat = 500\n",
    "\n",
    "# Generar transacciones normales por categoría\n",
    "transacciones = {}\n",
    "ids_transaccion = {}\n",
    "\n",
    "for i, cat in enumerate(categorias):\n",
    "    media, std = params_categorias[cat]\n",
    "    \n",
    "    # Generar montos normales\n",
    "    montos = np.random.normal(media, std, n_transacciones_por_cat)\n",
    "    montos = np.maximum(montos, 10)  # Mínimo $10\n",
    "    \n",
    "    # Inyectar anomalías (aproximadamente 3-5% del total)\n",
    "    n_anomalias_altas = np.random.randint(8, 15)\n",
    "    n_anomalias_bajas = np.random.randint(5, 10)\n",
    "    \n",
    "    # Anomalías altas (montos sospechosamente grandes)\n",
    "    indices_altas = np.random.choice(n_transacciones_por_cat, n_anomalias_altas, replace=False)\n",
    "    montos[indices_altas] = media + np.random.uniform(4, 8, n_anomalias_altas) * std\n",
    "    \n",
    "    # Anomalías bajas (posibles pruebas de tarjeta)\n",
    "    indices_bajas = np.random.choice(\n",
    "        [i for i in range(n_transacciones_por_cat) if i not in indices_altas],\n",
    "        n_anomalias_bajas, replace=False\n",
    "    )\n",
    "    montos[indices_bajas] = np.random.uniform(1, 15, n_anomalias_bajas)\n",
    "    \n",
    "    transacciones[cat] = montos\n",
    "    ids_transaccion[cat] = np.arange(i * 1000 + 1, i * 1000 + n_transacciones_por_cat + 1)\n",
    "\n",
    "# Crear arrays consolidados\n",
    "# Array 2D: (categorías, transacciones)\n",
    "montos_matriz = np.array([transacciones[cat] for cat in categorias])\n",
    "\n",
    "# Arrays 1D para análisis global\n",
    "todos_montos = np.concatenate([transacciones[cat] for cat in categorias])\n",
    "todas_categorias = np.concatenate([[cat] * n_transacciones_por_cat for cat in categorias])\n",
    "todos_ids = np.concatenate([ids_transaccion[cat] for cat in categorias])\n",
    "\n",
    "print(\"╔══════════════════════════════════════════════════════════════════╗\")\n",
    "print(\"║              DATOS DE TRANSACCIONES GENERADOS                    ║\")\n",
    "print(\"╠══════════════════════════════════════════════════════════════════╣\")\n",
    "print(f\"║  📊 montos_matriz    : shape {montos_matriz.shape}                      ║\")\n",
    "print(f\"║     (filas=categorías, columnas=transacciones)                 ║\")\n",
    "print(f\"║                                                                  ║\")\n",
    "print(f\"║  💳 todos_montos     : {len(todos_montos):,} transacciones totales          ║\")\n",
    "print(f\"║  🏷️  todas_categorias : {len(todas_categorias):,} etiquetas                    ║\")\n",
    "print(f\"║  🔢 todos_ids        : {len(todos_ids):,} identificadores                ║\")\n",
    "print(\"╚══════════════════════════════════════════════════════════════════╝\")\n",
    "\n",
    "print(\"\\n📍 Categorías disponibles:\")\n",
    "for i, cat in enumerate(categorias):\n",
    "    media, std = params_categorias[cat]\n",
    "    print(f\"   {i}: {cat:20s} (μ=${media:,}, σ=${std})\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## 🏋️ PARTE 1: Análisis Estadístico por Categoría (30 puntos)\n",
    "\n",
    "### Ejercicio 1.1: Estadísticas Descriptivas (10 puntos)\n",
    "\n",
    "Calcula las estadísticas básicas para cada categoría de transacción."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ═══════════════════════════════════════════════════════════════════\n",
    "#            EJERCICIO 1.1: ESTADÍSTICAS POR CATEGORÍA\n",
    "# ═══════════════════════════════════════════════════════════════════\n",
    "\n",
    "print(\"📊 ESTADÍSTICAS DESCRIPTIVAS POR CATEGORÍA\")\n",
    "print(\"═\" * 80)\n",
    "\n",
    "# TODO: Para cada categoría, calcula y almacena:\n",
    "# - Media\n",
    "# - Mediana\n",
    "# - Desviación estándar\n",
    "# - Mínimo y Máximo\n",
    "\n",
    "# Arrays para almacenar resultados\n",
    "medias = np.zeros(n_categorias)\n",
    "medianas = np.zeros(n_categorias)\n",
    "stds = np.zeros(n_categorias)\n",
    "minimos = np.zeros(n_categorias)\n",
    "maximos = np.zeros(n_categorias)\n",
    "\n",
    "for i, cat in enumerate(categorias):\n",
    "    datos = montos_matriz[i]  # Transacciones de esta categoría\n",
    "    \n",
    "    # TODO: Calcula las estadísticas\n",
    "    medias[i] = np.mean(datos)\n",
    "    medianas[i] = np.median(datos)\n",
    "    stds[i] = np.std(datos)\n",
    "    minimos[i] = np.min(datos)\n",
    "    maximos[i] = np.max(datos)\n",
    "\n",
    "# Mostrar resultados en tabla\n",
    "print(f\"\\n{'Categoría':<20} {'Media':>12} {'Mediana':>12} {'Std':>12} {'Mín':>10} {'Máx':>10}\")\n",
    "print(\"─\" * 80)\n",
    "\n",
    "for i, cat in enumerate(categorias):\n",
    "    print(f\"{cat:<20} ${medias[i]:>10,.2f} ${medianas[i]:>10,.2f} ${stds[i]:>10,.2f} ${minimos[i]:>8,.2f} ${maximos[i]:>8,.2f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Ejercicio 1.2: Cuartiles e IQR (10 puntos)\n",
    "\n",
    "Calcula los cuartiles y el rango intercuartílico para cada categoría."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ═══════════════════════════════════════════════════════════════════\n",
    "#                 EJERCICIO 1.2: CUARTILES E IQR\n",
    "# ═══════════════════════════════════════════════════════════════════\n",
    "\n",
    "print(\"📈 CUARTILES E IQR POR CATEGORÍA\")\n",
    "print(\"═\" * 80)\n",
    "\n",
    "# Arrays para almacenar resultados\n",
    "Q1_arr = np.zeros(n_categorias)\n",
    "Q2_arr = np.zeros(n_categorias)\n",
    "Q3_arr = np.zeros(n_categorias)\n",
    "IQR_arr = np.zeros(n_categorias)\n",
    "\n",
    "for i, cat in enumerate(categorias):\n",
    "    datos = montos_matriz[i]\n",
    "    \n",
    "    # TODO: Calcula Q1, Q2 (mediana), Q3 e IQR\n",
    "    Q1_arr[i] = np.percentile(datos, 25)\n",
    "    Q2_arr[i] = np.percentile(datos, 50)\n",
    "    Q3_arr[i] = np.percentile(datos, 75)\n",
    "    IQR_arr[i] = Q3_arr[i] - Q1_arr[i]\n",
    "\n",
    "# Mostrar resultados\n",
    "print(f\"\\n{'Categoría':<20} {'Q1 (25%)':>12} {'Q2 (50%)':>12} {'Q3 (75%)':>12} {'IQR':>12}\")\n",
    "print(\"─\" * 72)\n",
    "\n",
    "for i, cat in enumerate(categorias):\n",
    "    print(f\"{cat:<20} ${Q1_arr[i]:>10,.2f} ${Q2_arr[i]:>10,.2f} ${Q3_arr[i]:>10,.2f} ${IQR_arr[i]:>10,.2f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Ejercicio 1.3: Límites para Outliers (10 puntos)\n",
    "\n",
    "Calcula los límites inferior y superior para detectar outliers usando el método IQR."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ═══════════════════════════════════════════════════════════════════\n",
    "#               EJERCICIO 1.3: LÍMITES PARA OUTLIERS\n",
    "# ═══════════════════════════════════════════════════════════════════\n",
    "\n",
    "print(\"🚧 LÍMITES PARA DETECCIÓN DE OUTLIERS (Método IQR)\")\n",
    "print(\"═\" * 80)\n",
    "\n",
    "# Factor IQR estándar\n",
    "FACTOR_IQR = 1.5\n",
    "\n",
    "# Arrays para almacenar límites\n",
    "limites_inf = np.zeros(n_categorias)\n",
    "limites_sup = np.zeros(n_categorias)\n",
    "\n",
    "for i, cat in enumerate(categorias):\n",
    "    # TODO: Calcula los límites usando Q1, Q3 e IQR\n",
    "    # Límite inferior = Q1 - 1.5 * IQR\n",
    "    # Límite superior = Q3 + 1.5 * IQR\n",
    "    \n",
    "    limites_inf[i] = Q1_arr[i] - 1.5 * IQR_arr[i]\n",
    "    limites_sup[i] = Q3_arr[i] + 1.5 * IQR_arr[i]\n",
    "\n",
    "# Mostrar resultados\n",
    "print(f\"\\n{'Categoría':<20} {'Límite Inf':>15} {'Límite Sup':>15} {'Rango Válido':>20}\")\n",
    "print(\"─\" * 75)\n",
    "\n",
    "for i, cat in enumerate(categorias):\n",
    "    # El límite inferior no puede ser negativo para montos\n",
    "    lim_inf_real = max(0, limites_inf[i])\n",
    "    rango = f\"${lim_inf_real:,.0f} - ${limites_sup[i]:,.0f}\"\n",
    "    print(f\"{cat:<20} ${limites_inf[i]:>13,.2f} ${limites_sup[i]:>13,.2f} {rango:>20}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## 🏋️ PARTE 2: Detección de Outliers con IQR (25 puntos)\n",
    "\n",
    "### Ejercicio 2.1: Identificar Outliers por Categoría (15 puntos)\n",
    "\n",
    "Detecta las transacciones anómalas en cada categoría usando el método IQR."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ═══════════════════════════════════════════════════════════════════\n",
    "#            EJERCICIO 2.1: DETECCIÓN DE OUTLIERS CON IQR\n",
    "# ═══════════════════════════════════════════════════════════════════\n",
    "\n",
    "print(\"🔍 DETECCIÓN DE TRANSACCIONES ANÓMALAS (Método IQR)\")\n",
    "print(\"═\" * 80)\n",
    "\n",
    "# Diccionario para almacenar resultados\n",
    "outliers_iqr = {}\n",
    "n_outliers_iqr = np.zeros(n_categorias, dtype=int)\n",
    "\n",
    "for i, cat in enumerate(categorias):\n",
    "    datos = montos_matriz[i]\n",
    "    ids = ids_transaccion[cat]\n",
    "    \n",
    "    # TODO: Crea una máscara booleana para identificar outliers\n",
    "    # Un valor es outlier si: valor < limite_inferior OR valor > limite_superior\n",
    "    \n",
    "    mascara_outliers = (datos < limites_inf[i]) | (datos > limites_sup[i])\n",
    "    \n",
    "    # Separar outliers inferiores y superiores\n",
    "    mascara_inf = datos < limites_inf[i]\n",
    "    mascara_sup = datos > limites_sup[i]\n",
    "    \n",
    "    # Almacenar resultados\n",
    "    outliers_iqr[cat] = {\n",
    "        'ids': ids[mascara_outliers],\n",
    "        'montos': datos[mascara_outliers],\n",
    "        'n_total': np.sum(mascara_outliers),\n",
    "        'n_inferiores': np.sum(mascara_inf),\n",
    "        'n_superiores': np.sum(mascara_sup)\n",
    "    }\n",
    "    n_outliers_iqr[i] = np.sum(mascara_outliers)\n",
    "\n",
    "# Mostrar resumen\n",
    "print(f\"\\n{'Categoría':<20} {'Total Trans.':>12} {'Outliers':>10} {'% Anomalías':>12} {'Inf.':>8} {'Sup.':>8}\")\n",
    "print(\"─\" * 75)\n",
    "\n",
    "for i, cat in enumerate(categorias):\n",
    "    pct = (n_outliers_iqr[i] / n_transacciones_por_cat) * 100\n",
    "    info = outliers_iqr[cat]\n",
    "    print(f\"{cat:<20} {n_transacciones_por_cat:>12,} {info['n_total']:>10} {pct:>11.1f}% {info['n_inferiores']:>8} {info['n_superiores']:>8}\")\n",
    "\n",
    "print(f\"\\n📊 Total de outliers detectados: {np.sum(n_outliers_iqr)}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Ejercicio 2.2: Análisis de Outliers Detectados (10 puntos)\n",
    "\n",
    "Analiza las características de los outliers detectados."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ═══════════════════════════════════════════════════════════════════\n",
    "#              EJERCICIO 2.2: ANÁLISIS DE OUTLIERS IQR\n",
    "# ═══════════════════════════════════════════════════════════════════\n",
    "\n",
    "print(\"📋 ANÁLISIS DETALLADO DE OUTLIERS (Método IQR)\")\n",
    "print(\"═\" * 80)\n",
    "\n",
    "for cat in categorias:\n",
    "    info = outliers_iqr[cat]\n",
    "    \n",
    "    if info['n_total'] > 0:\n",
    "        montos_out = info['montos']\n",
    "        \n",
    "        # TODO: Calcula estadísticas de los outliers\n",
    "        monto_min_outlier = np.min(montos_out)\n",
    "        monto_max_outlier = np.max(montos_out)\n",
    "        monto_promedio_outlier = np.mean(montos_out)\n",
    "        \n",
    "        print(f\"\\n🏷️  {cat}\")\n",
    "        print(f\"   Outliers detectados: {info['n_total']}\")\n",
    "        print(f\"   Monto mínimo outlier: ${monto_min_outlier:,.2f}\")\n",
    "        print(f\"   Monto máximo outlier: ${monto_max_outlier:,.2f}\")\n",
    "        print(f\"   Monto promedio outlier: ${monto_promedio_outlier:,.2f}\")\n",
    "        \n",
    "        # Mostrar los 3 outliers más extremos (más altos)\n",
    "        if info['n_superiores'] > 0:\n",
    "            idx_ordenados = np.argsort(montos_out)[::-1]  # Descendente\n",
    "            print(f\"   Top 3 montos más altos:\")\n",
    "            for j in range(min(3, len(idx_ordenados))):\n",
    "                idx = idx_ordenados[j]\n",
    "                print(f\"      - ID {info['ids'][idx]}: ${montos_out[idx]:,.2f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## 🏋️ PARTE 3: Detección de Outliers con Z-Score (25 puntos)\n",
    "\n",
    "### Ejercicio 3.1: Calcular Z-Scores (10 puntos)\n",
    "\n",
    "Calcula el Z-Score para cada transacción."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ═══════════════════════════════════════════════════════════════════\n",
    "#               EJERCICIO 3.1: CÁLCULO DE Z-SCORES\n",
    "# ═══════════════════════════════════════════════════════════════════\n",
    "\n",
    "print(\"📐 CÁLCULO DE Z-SCORES POR CATEGORÍA\")\n",
    "print(\"═\" * 80)\n",
    "\n",
    "# Umbral para considerar outlier\n",
    "UMBRAL_ZSCORE = 3\n",
    "\n",
    "# Matriz para almacenar z-scores\n",
    "zscores_matriz = np.zeros_like(montos_matriz)\n",
    "\n",
    "for i, cat in enumerate(categorias):\n",
    "    datos = montos_matriz[i]\n",
    "    \n",
    "    # TODO: Calcula el Z-Score para cada transacción\n",
    "    # Fórmula: z = (x - media) / std\n",
    "    \n",
    "    media_cat = np.mean(datos)\n",
    "    std_cat = np.std(datos)\n",
    "    zscores_matriz[i] = (datos - media_cat) / std_cat\n",
    "\n",
    "# Verificar cálculo mostrando estadísticas de z-scores\n",
    "print(f\"\\n{'Categoría':<20} {'Media Z':>10} {'Std Z':>10} {'Min Z':>10} {'Max Z':>10}\")\n",
    "print(\"─\" * 65)\n",
    "\n",
    "for i, cat in enumerate(categorias):\n",
    "    zs = zscores_matriz[i]\n",
    "    print(f\"{cat:<20} {np.mean(zs):>10.4f} {np.std(zs):>10.4f} {np.min(zs):>10.2f} {np.max(zs):>10.2f}\")\n",
    "\n",
    "print(f\"\\n💡 Nota: La media de Z-scores debe ser ~0 y la std ~1\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Ejercicio 3.2: Detectar Outliers con Z-Score (15 puntos)\n",
    "\n",
    "Identifica outliers donde |Z-Score| > 3."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ═══════════════════════════════════════════════════════════════════\n",
    "#          EJERCICIO 3.2: DETECCIÓN DE OUTLIERS CON Z-SCORE\n",
    "# ═══════════════════════════════════════════════════════════════════\n",
    "\n",
    "print(f\"🔍 DETECCIÓN DE OUTLIERS CON Z-SCORE (umbral = {UMBRAL_ZSCORE})\")\n",
    "print(\"═\" * 80)\n",
    "\n",
    "# Diccionario para almacenar resultados\n",
    "outliers_zscore = {}\n",
    "n_outliers_zscore = np.zeros(n_categorias, dtype=int)\n",
    "\n",
    "for i, cat in enumerate(categorias):\n",
    "    datos = montos_matriz[i]\n",
    "    zscores = zscores_matriz[i]\n",
    "    ids = ids_transaccion[cat]\n",
    "    \n",
    "    # TODO: Crea una máscara para outliers donde |z-score| > umbral\n",
    "    mascara_outliers_z = np.abs(zscores) > UMBRAL_ZSCORE\n",
    "    \n",
    "    # Clasificar por tipo\n",
    "    mascara_z_neg = zscores < -UMBRAL_ZSCORE  # Muy bajos\n",
    "    mascara_z_pos = zscores > UMBRAL_ZSCORE   # Muy altos\n",
    "    \n",
    "    outliers_zscore[cat] = {\n",
    "        'ids': ids[mascara_outliers_z],\n",
    "        'montos': datos[mascara_outliers_z],\n",
    "        'zscores': zscores[mascara_outliers_z],\n",
    "        'n_total': np.sum(mascara_outliers_z),\n",
    "        'n_bajos': np.sum(mascara_z_neg),\n",
    "        'n_altos': np.sum(mascara_z_pos)\n",
    "    }\n",
    "    n_outliers_zscore[i] = np.sum(mascara_outliers_z)\n",
    "\n",
    "# Mostrar resumen\n",
    "print(f\"\\n{'Categoría':<20} {'Total Trans.':>12} {'Outliers':>10} {'% Anomalías':>12} {'Z<-3':>8} {'Z>3':>8}\")\n",
    "print(\"─\" * 75)\n",
    "\n",
    "for i, cat in enumerate(categorias):\n",
    "    pct = (n_outliers_zscore[i] / n_transacciones_por_cat) * 100\n",
    "    info = outliers_zscore[cat]\n",
    "    print(f\"{cat:<20} {n_transacciones_por_cat:>12,} {info['n_total']:>10} {pct:>11.1f}% {info['n_bajos']:>8} {info['n_altos']:>8}\")\n",
    "\n",
    "print(f\"\\n📊 Total de outliers detectados (Z-Score): {np.sum(n_outliers_zscore)}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## 🏋️ PARTE 4: Comparación y Reporte Final (20 puntos)\n",
    "\n",
    "### Ejercicio 4.1: Comparar Métodos (10 puntos)\n",
    "\n",
    "Compara los resultados de ambos métodos de detección."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ═══════════════════════════════════════════════════════════════════\n",
    "#               EJERCICIO 4.1: COMPARACIÓN DE MÉTODOS\n",
    "# ═══════════════════════════════════════════════════════════════════\n",
    "\n",
    "print(\"🔄 COMPARACIÓN DE MÉTODOS DE DETECCIÓN\")\n",
    "print(\"═\" * 80)\n",
    "\n",
    "# TODO: Calcula métricas de comparación\n",
    "\n",
    "# Total de outliers por método\n",
    "total_iqr = np.sum(n_outliers_iqr)\n",
    "total_zscore = np.sum(n_outliers_zscore)\n",
    "\n",
    "print(f\"\\n📊 RESUMEN GLOBAL:\")\n",
    "print(f\"   Método IQR:     {total_iqr} outliers detectados\")\n",
    "print(f\"   Método Z-Score: {total_zscore} outliers detectados\")\n",
    "\n",
    "# Comparación por categoría\n",
    "print(f\"\\n{'Categoría':<20} {'IQR':>10} {'Z-Score':>10} {'Diferencia':>12} {'Coincidencia':>15}\")\n",
    "print(\"─\" * 72)\n",
    "\n",
    "for i, cat in enumerate(categorias):\n",
    "    n_iqr = n_outliers_iqr[i]\n",
    "    n_zs = n_outliers_zscore[i]\n",
    "    diff = n_iqr - n_zs\n",
    "    \n",
    "    # TODO: Calcula cuántos outliers coinciden en ambos métodos\n",
    "    # (están en ambas listas de outliers)\n",
    "    ids_iqr = set(outliers_iqr[cat]['ids'])\n",
    "    ids_zscore = set(outliers_zscore[cat]['ids'])\n",
    "    coincidentes = len(ids_iqr & ids_zscore)  # Intersección\n",
    "    \n",
    "    print(f\"{cat:<20} {n_iqr:>10} {n_zs:>10} {diff:>+12} {coincidentes:>15}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Ejercicio 4.2: Reporte de Transacciones Sospechosas (10 puntos)\n",
    "\n",
    "Genera un reporte final de las transacciones más sospechosas."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ═══════════════════════════════════════════════════════════════════\n",
    "#            EJERCICIO 4.2: REPORTE DE TRANSACCIONES SOSPECHOSAS\n",
    "# ═══════════════════════════════════════════════════════════════════\n",
    "\n",
    "print(\"╔══════════════════════════════════════════════════════════════════════════╗\")\n",
    "print(\"║                                                                          ║\")\n",
    "print(\"║        🚨 SECUREBANK - REPORTE DE TRANSACCIONES SOSPECHOSAS 🚨           ║\")\n",
    "print(\"║                                                                          ║\")\n",
    "print(\"╠══════════════════════════════════════════════════════════════════════════╣\")\n",
    "\n",
    "# Identificar transacciones detectadas por AMBOS métodos (alta confianza)\n",
    "print(\"║                                                                          ║\")\n",
    "print(\"║  ⚠️  ALTA PRIORIDAD (detectadas por ambos métodos)                       ║\")\n",
    "print(\"║  ─────────────────────────────────────────────────────────────────────   ║\")\n",
    "\n",
    "total_alta_prioridad = 0\n",
    "\n",
    "for cat in categorias:\n",
    "    ids_iqr = set(outliers_iqr[cat]['ids'])\n",
    "    ids_zscore = set(outliers_zscore[cat]['ids'])\n",
    "    \n",
    "    # TODO: Encuentra las transacciones que están en AMBOS conjuntos\n",
    "    ids_ambos = ids_iqr & ids_zscore\n",
    "    \n",
    "    if len(ids_ambos) > 0:\n",
    "        total_alta_prioridad += len(ids_ambos)\n",
    "        \n",
    "        # Obtener montos de estas transacciones\n",
    "        for trans_id in list(ids_ambos)[:3]:  # Mostrar máximo 3\n",
    "            idx = np.where(ids_transaccion[cat] == trans_id)[0][0]\n",
    "            monto = montos_matriz[categorias.index(cat), idx]\n",
    "            zscore = zscores_matriz[categorias.index(cat), idx]\n",
    "            print(f\"║    ID {trans_id}: {cat:15s} ${monto:>10,.2f} (Z={zscore:+.2f}){'':>10}║\")\n",
    "\n",
    "print(\"║                                                                          ║\")\n",
    "print(f\"║  📊 RESUMEN EJECUTIVO                                                    ║\")\n",
    "print(\"║  ─────────────────────────────────────────────────────────────────────   ║\")\n",
    "\n",
    "# TODO: Calcula estadísticas finales\n",
    "total_transacciones = len(todos_montos)\n",
    "total_outliers_unicos = len(set.union(*[set(outliers_iqr[c]['ids']) | set(outliers_zscore[c]['ids']) for c in categorias]))  # Unión de ambos métodos\n",
    "pct_anomalias = (total_outliers_unicos / total_transacciones) * 100\n",
    "\n",
    "print(f\"║    Total transacciones analizadas:    {total_transacciones:>6,}{'':>25}║\")\n",
    "print(f\"║    Transacciones sospechosas:         {total_outliers_unicos:>6,} ({pct_anomalias:.1f}%){'':>17}║\")\n",
    "print(f\"║    Alta prioridad (ambos métodos):    {total_alta_prioridad:>6,}{'':>25}║\")\n",
    "print(\"║                                                                          ║\")\n",
    "print(\"╚══════════════════════════════════════════════════════════════════════════╝\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## 🎯 EJERCICIO BONUS: Análisis de Correlación (10 puntos)\n",
    "\n",
    "Analiza si existe correlación entre categorías en los patrones de gasto."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ═══════════════════════════════════════════════════════════════════\n",
    "#                  EJERCICIO BONUS: CORRELACIÓN\n",
    "# ═══════════════════════════════════════════════════════════════════\n",
    "\n",
    "print(\"📈 ANÁLISIS DE CORRELACIÓN ENTRE CATEGORÍAS\")\n",
    "print(\"═\" * 70)\n",
    "\n",
    "# TODO: Calcula la matriz de correlación entre categorías\n",
    "# Usa montos_matriz donde cada fila es una categoría\n",
    "\n",
    "matriz_correlacion = np.corrcoef(montos_matriz)\n",
    "\n",
    "# Mostrar matriz de correlación\n",
    "print(f\"\\n{'':>18}\", end='')\n",
    "for cat in categorias:\n",
    "    print(f\"{cat[:8]:>10}\", end='')\n",
    "print()\n",
    "print(\"─\" * 70)\n",
    "\n",
    "for i, cat in enumerate(categorias):\n",
    "    print(f\"{cat:<18}\", end='')\n",
    "    for j in range(n_categorias):\n",
    "        valor = matriz_correlacion[i, j]\n",
    "        if i == j:\n",
    "            print(f\"{'1.00':>10}\", end='')\n",
    "        else:\n",
    "            print(f\"{valor:>10.3f}\", end='')\n",
    "    print()\n",
    "\n",
    "# Encontrar las correlaciones más fuertes (excluyendo diagonal)\n",
    "print(f\"\\n🔗 CORRELACIONES MÁS FUERTES:\")\n",
    "print(\"─\" * 50)\n",
    "\n",
    "# TODO: Identifica el par de categorías con mayor correlación\n",
    "# (excluyendo la diagonal)\n",
    "max_corr = 0\n",
    "par_max = ('', '')\n",
    "\n",
    "for i in range(n_categorias):\n",
    "    for j in range(i+1, n_categorias):\n",
    "        if abs(matriz_correlacion[i, j]) > abs(max_corr):\n",
    "            max_corr = matriz_correlacion[i, j]\n",
    "            par_max = (categorias[i], categorias[j])\n",
    "\n",
    "print(f\"   Mayor correlación: {par_max[0]} ↔ {par_max[1]}\")\n",
    "print(f\"   Valor: {max_corr:.4f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## ✅ Checklist de Entrega\n",
    "\n",
    "Antes de entregar, verifica que:\n",
    "\n",
    "| # | Criterio | Puntos |\n",
    "|---|----------|--------|\n",
    "| 1 | Parte 1: Análisis Estadístico por Categoría | 30 |\n",
    "| 2 | Parte 2: Detección de Outliers con IQR | 25 |\n",
    "| 3 | Parte 3: Detección de Outliers con Z-Score | 25 |\n",
    "| 4 | Parte 4: Comparación y Reporte Final | 20 |\n",
    "| 5 | BONUS: Análisis de Correlación | +10 |\n",
    "| | **Total posible** | **110** |\n",
    "\n",
    "### 📝 Notas Importantes\n",
    "\n",
    "- Usa operaciones vectorizadas de NumPy (evita loops donde sea posible)\n",
    "- Verifica que tus Z-Scores tengan media ~0 y std ~1\n",
    "- El código debe ejecutarse sin errores de principio a fin\n",
    "- Los outliers detectados deben ser razonables (3-5% típicamente)\n",
    "\n",
    "---\n",
    "\n",
    "### 🏅 Criterios de Evaluación\n",
    "\n",
    "```\n",
    "RÚBRICA DE CALIFICACIÓN\n",
    "═══════════════════════════════════════════════════════\n",
    "\n",
    "✓ Cálculos estadísticos correctos         → 40%\n",
    "✓ Detección de outliers precisa           → 30%\n",
    "✓ Comparación de métodos coherente        → 15%\n",
    "✓ Código limpio y vectorizado             → 10%\n",
    "✓ Interpretaciones razonables             → 5%\n",
    "\n",
    "═══════════════════════════════════════════════════════\n",
    "```\n",
    "\n",
    "---\n",
    "\n",
    "*SecureBank Fraud Detection - Reto desarrollado para Programación para Ciencia de Datos, IPN 2026*"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3 (ipykernel)",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.11.9"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}

: 